In [108]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import imageio
from datetime import datetime, timedelta
import geopandas as gpd
from shapely.geometry import Point

# Assuming your data is in a CSV file with columns: 'IncidentDate', 'Latitude', 'Longitude'
data = pd.read_csv('../data/cleaned_2021-jan2025.csv')
data.dropna(subset=['Latitude', 'Longitude'])
data['IncidentDate'] = pd.to_datetime(data['IncidentDate'], format="mixed", dayfirst=False)

len(data)

C:\Users\jz043x\AppData\Local\Temp\ipykernel_4632\4068511443.py:10: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('../data/cleaned_2021-jan2025.csv')


228776

In [109]:
# Filter data for the year 2024
data = data[(data['IncidentDate'] >= '2024-01-01') & (data['IncidentDate'] < '2025-01-01')]
len(data)

49662

In [110]:
# REMOVE DATA OUTSIDE CITY BOUNDS

# Load the neighborhood shapefile
city_bounds = gpd.read_file('../data/city_bounds/stl_boundary.shp')


points = gpd.GeoDataFrame(data, geometry=gpd.points_from_xy(data.Longitude, data.Latitude))


# points = points.to_crs(city_bounds.crs)
points.set_crs(epsg=4326, inplace=True)

if city_bounds.crs != points.crs:
    points = points.to_crs(city_bounds.crs)

within_points = gpd.sjoin(points, city_bounds, how='inner', predicate = 'within')

columns_to_drop = ['index_right'] + list(city_bounds.columns)

within_points = within_points.drop(columns=columns_to_drop)

print(len(within_points))
data = within_points


48915


In [94]:
# Define the grid size (e.g., 0.01 degrees for a medium-sized city)
lat_min, lat_max = data['Latitude'].min(), data['Latitude'].max()
lon_min, lon_max = data['Longitude'].min(), data['Longitude'].max()
lat_bins = np.arange(lat_min, lat_max, 0.003)
lon_bins = np.arange(lon_min, lon_max, 0.003)

print(lat_min, lat_max, len(lat_bins))
print(lon_min, lon_max, len(lon_bins))

38.537907 38.771904 78
-90.318869 -90.178843 47


In [95]:
# Create a function to aggregate data into grid cells for a given week
def aggregate_weekly_data(data, week_start, week_end):
    weekly_data = data[(data['IncidentDate'] >= week_start) & (data['IncidentDate'] < week_end)]
    heatmap, _, _ = np.histogram2d(weekly_data['Latitude'], weekly_data['Longitude'], bins=[lat_bins, lon_bins])
    return heatmap

# Generate heatmaps for each week of 2024
start_date = datetime(2024, 1, 1)
weeks = [(start_date + timedelta(weeks=w), start_date + timedelta(weeks=w+1)) for w in range(52)]
heatmaps = [aggregate_weekly_data(data, week_start, week_end) for week_start, week_end in weeks]


In [104]:
#Filter for firearm crime

# Generate heatmaps for each week of 2024
start_date = datetime(2024, 1, 1)
weeks = [(start_date + timedelta(weeks=w), start_date + timedelta(weeks=w+1)) for w in range(52)]
heatmaps = [aggregate_weekly_data(data[data['FirearmUsed'] == 'Yes'], week_start, week_end) for week_start, week_end in weeks]


In [116]:
#Filter for theft crimes

# Generate heatmaps for each week of 2024
start_date = datetime(2024, 1, 1)
weeks = [(start_date + timedelta(weeks=w), start_date + timedelta(weeks=w+1)) for w in range(52)]
heatmaps = [aggregate_weekly_data(data[data['NIBRS'].str.startswith('23')], week_start, week_end) for week_start, week_end in weeks]



In [117]:

# Create a GIF from the weekly heatmaps with consistent colorbar scale
filenames = []

# Determine the global min and max for the color scale
global_min = np.min([heatmap.min() for heatmap in heatmaps])
global_max = np.max([heatmap.max() for heatmap in heatmaps])

for i, heatmap in enumerate(heatmaps):
    plt.figure(figsize=(8, 6))
    plt.imshow(heatmap.T, origin='lower', aspect='auto', cmap='hot', interpolation='nearest', vmin=global_min, vmax=global_max)
    plt.colorbar(label='Occurrences')
    plt.title(f'Week {i+1}')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    filename = f'heatmap_week_{i+1}.png'
    plt.savefig(filename)
    filenames.append(filename)
    plt.close()

# Create a GIF
with imageio.get_writer('theft_heatmap_animation.gif', mode='I', duration=0.5) as writer:
    for filename in filenames:
        image = imageio.imread(filename)
        writer.append_data(image)

# Cleanup generated images
import os
for filename in filenames:
    os.remove(filename)

print("GIF created successfully: heatmap_animation.gif")

C:\Users\jz043x\AppData\Local\Temp\ipykernel_4632\1889467888.py:26: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  image = imageio.imread(filename)


GIF created successfully: heatmap_animation.gif
